# Chronological split design for M0 — one shared P95-led outer split

This notebook is the sole owner of the outer split. It evaluates date cutoffs using the P95 primary outcome, selects one cutoff based on row/class balance and test-horizon support, then writes a master split CSV keyed by `row_id`. P90 receives exactly the same assignments. Household overlap is reported as `test_seen` and `test_unseen`; the natural temporal split is not a random household holdout.


## Stage 0 — configuration and input contracts

The candidate score follows the plan: sum the absolute differences between achieved development fractions for rows, P95 negatives, and P95 positives and the desired development fraction. Candidate outer cutoffs must also leave enough test calendar span and positive events for both P95 and P90. Within the selected outer development period, this notebook assigns whole dates to model development, calibration, and threshold selection using cumulative row counts near 70%/15%/15%. The final proportions and calendar spans are reported so their trade-offs are visible.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir(): raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")
PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "04_split_design"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "04_split_design"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CANONICAL_PATH = PROCESSED / "05_model_ready_m0.csv"
P95_PATH = PROCESSED / "05_model_ready_m0_p95.csv"
P90_PATH = PROCESSED / "05_model_ready_m0_p90.csv"
CANONICAL_CONTRACT_PATH = PROCESSED / "05_model_ready_m0_dtype_contract.json"
P95_CONTRACT_PATH = PROCESSED / "05_model_ready_m0_p95_dtype_contract.json"
P90_CONTRACT_PATH = PROCESSED / "05_model_ready_m0_p90_dtype_contract.json"
FEATURE_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "04_feature_engineering" / "04_run_manifest.json"
MASTER_SPLIT_PATH = PROCESSED / "04_master_split.csv"
MASTER_SPLIT_CONTRACT_PATH = PROCESSED / "04_master_split_dtype_contract.json"

DESIRED_DEVELOPMENT_FRACTIONS = [0.80, 0.75, 0.70]
MIN_TEST_CALENDAR_DAYS = 35
MIN_TEST_POSITIVE_EVENTS = 100
INTERNAL_BLOCK_TARGET_FRACTIONS = {"model_development": 0.70, "calibration": 0.15, "threshold_selection": 0.15}
MIN_INTERNAL_BLOCK_DATES = 5
MIN_INTERNAL_BLOCK_EVENTS = 100
MIN_INTERNAL_BLOCK_HOUSEHOLDS = 100
CSV_CHUNK_SIZE = 100_000

def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size): digest.update(block)
    return digest.hexdigest()

def read_contract_csv(path, contract, usecols=None):
    selected = list(contract["columns"]) if usecols is None else list(usecols)
    dates = [name for name in contract.get("date_columns", []) if name in selected]
    dtypes = {name: dtype for name, dtype in contract["columns"].items() if name in selected and dtype != "datetime"}
    return pd.read_csv(path, usecols=selected, dtype=dtypes or None, parse_dates=dates or None,
                       true_values=contract["reader"]["true_values"], false_values=contract["reader"]["false_values"],
                       keep_default_na=contract["reader"]["keep_default_na"], na_values=contract["reader"]["na_values"],
                       encoding=contract["encoding"], sep=contract["separator"], low_memory=False)

def write_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def write_csv_atomic(frame, path):
    path = Path(path)
    temporary = path.with_name(path.stem + ".tmp" + path.suffix)
    if temporary.exists(): temporary.unlink()
    frame.to_csv(temporary, index=False, encoding="utf-8", na_rep="", date_format="%Y-%m-%d %H:%M:%S", float_format="%.17g")
    temporary.replace(path)

def dtype_name(dtype):
    if pd.api.types.is_datetime64_any_dtype(dtype): return "datetime"
    if pd.api.types.is_bool_dtype(dtype): return "boolean"
    if pd.api.types.is_integer_dtype(dtype): return str(dtype)
    if pd.api.types.is_float_dtype(dtype): return str(dtype)
    return "string"

def write_dtype_contract(frame, path, contract_path):
    dtypes = {name: dtype_name(frame[name].dtype) for name in frame.columns}
    contract = {"contract_version": 1, "path": str(path), "format": "csv", "encoding": "utf-8", "separator": ",", "missing_value": "",
                "date_columns": [name for name, dtype in dtypes.items() if dtype == "datetime"], "columns": dtypes,
                "reader": {"true_values": ["True"], "false_values": ["False"], "keep_default_na": False, "na_values": [""]}}
    write_json(contract, contract_path)
    return contract

def save_table(frame, filename, title=None):
    path = TABLE_DIR / filename
    frame.to_csv(path, index=False, encoding="utf-8", na_rep="")
    if title: print(f"{title}: {path.name}")
    display(frame.head(40))
    return path

required = [CANONICAL_PATH, P95_PATH, P90_PATH, CANONICAL_CONTRACT_PATH, P95_CONTRACT_PATH, P90_CONTRACT_PATH, FEATURE_MANIFEST_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing: raise FileNotFoundError(f"Split design inputs missing: {missing}")
feature_manifest = read_json(FEATURE_MANIFEST_PATH)
contracts = {"canonical": read_json(CANONICAL_CONTRACT_PATH), "p95": read_json(P95_CONTRACT_PATH), "p90": read_json(P90_CONTRACT_PATH)}
paths = {"canonical_model_dataset": CANONICAL_PATH, "p95_model_dataset": P95_PATH, "p90_model_dataset": P90_PATH}
expected_signatures = feature_manifest["outputs"]
for key, path in paths.items():
    actual_hash = sha256(path)
    expected = expected_signatures[key]
    if actual_hash != expected["sha256"] or path.stat().st_size != expected["size_bytes"]:
        raise RuntimeError(f"Input {path.name} does not match feature-engineering manifest")
    if pd.read_csv(path, nrows=0).columns.tolist() != list(contracts[key.removesuffix("_model_dataset")]["columns"]):
        raise RuntimeError(f"{path.name} header differs from its dtype contract")
print("Verified canonical/P95/P90 model-ready CSV signatures and dtype contracts.")


## Stage 1 — compare ordered rows and build candidate cutoff diagnostics

Each candidate date is inclusive in development; all later target dates are held out. Candidate cutoffs are computed from unique target dates, never by sorting or splitting individual rows. P95 alone determines the cutoff; P90 is evaluated with that same date afterward.


In [ ]:
KEY_COLUMNS = ["row_id", "household_id", "measurement_date"]
p95 = read_contract_csv(P95_PATH, contracts["p95"], usecols=KEY_COLUMNS + ["target"])
p90 = read_contract_csv(P90_PATH, contracts["p90"], usecols=KEY_COLUMNS + ["target"])
canonical_contract = contracts["canonical"]
canonical = read_contract_csv(CANONICAL_PATH, canonical_contract, usecols=KEY_COLUMNS + ["target_high_demand_p95", "target_high_demand_p90"])
for frame in (p95, p90, canonical):
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()

input_checks = pd.DataFrame([
    ("same_population_lengths", len(p95) == len(p90) == len(canonical), f"{len(p95):,}/{len(p90):,}/{len(canonical):,}"),
    ("same_ordered_row_ids", p95.row_id.equals(p90.row_id) and p95.row_id.equals(canonical.row_id), "P95=P90=canonical"),
    ("same_ordered_household_date_keys", p95[KEY_COLUMNS[1:]].equals(p90[KEY_COLUMNS[1:]]) and p95[KEY_COLUMNS[1:]].equals(canonical[KEY_COLUMNS[1:]]), "same household and target date"),
    ("unique_model_rows", all(frame.row_id.is_unique for frame in (p95, p90, canonical)), "row_id unique"),
    ("P95_primary_binary", p95.target.isin([0, 1]).all(), str(p95.target.value_counts().to_dict())),
    ("P90_sensitivity_binary", p90.target.isin([0, 1]).all(), str(p90.target.value_counts().to_dict())),
    ("same_label_as_canonical", p95.target.equals(canonical.target_high_demand_p95.astype("int8")) and p90.target.equals(canonical.target_high_demand_p90.astype("int8")), "exports reproduce canonical labels"),
    ("no_prior_split_present", not any(name in canonical.columns for name in ["split_name", "outer_split"]), "one outer split will be assigned here"),
], columns=["check", "passed", "evidence"])
save_table(input_checks, "04_01_split_input_checks.csv", "Split input checks")
if not input_checks.passed.all(): raise RuntimeError(f"Split input validation failed: {input_checks.loc[~input_checks.passed, 'check'].tolist()}")

date_counts = p95.groupby("measurement_date", as_index=False).agg(
    rows=("row_id", "size"), p95_positive=("target", "sum"), households=("household_id", "nunique"))
p90_date_counts = p90.groupby("measurement_date", as_index=False).agg(p90_positive=("target", "sum"))
date_counts = date_counts.merge(p90_date_counts, on="measurement_date", how="left", validate="one_to_one")
if date_counts["p90_positive"].isna().any():
    raise RuntimeError("P90 event counts are missing for one or more P95 dates")
date_counts["p90_positive"] = date_counts["p90_positive"].astype("int64")
date_counts = date_counts.sort_values("measurement_date").reset_index(drop=True)
date_counts["p95_negative"] = date_counts["rows"] - date_counts["p95_positive"]
date_counts["p90_negative"] = date_counts["rows"] - date_counts["p90_positive"]
date_counts["cum_rows"] = date_counts["rows"].cumsum()
date_counts["cum_p95_positive"] = date_counts["p95_positive"].cumsum()
date_counts["cum_p95_negative"] = date_counts["p95_negative"].cumsum()
date_counts["cum_p90_positive"] = date_counts["p90_positive"].cumsum()
total_rows = int(date_counts.rows.sum())
total_positive = int(date_counts.p95_positive.sum())
total_negative = int(date_counts.p95_negative.sum())
total_p90_positive = int(date_counts.p90_positive.sum())

candidate_rows = []
for desired_fraction in DESIRED_DEVELOPMENT_FRACTIONS:
    candidate = date_counts.copy()
    candidate["dev_row_fraction"] = candidate["cum_rows"] / total_rows
    candidate["dev_positive_fraction"] = candidate["cum_p95_positive"] / total_positive
    candidate["dev_negative_fraction"] = candidate["cum_p95_negative"] / total_negative
    candidate["score"] = (candidate["dev_row_fraction"] - desired_fraction).abs() + (candidate["dev_positive_fraction"] - desired_fraction).abs() + (candidate["dev_negative_fraction"] - desired_fraction).abs()
    candidate_index = int(candidate["score"].idxmin())
    row = candidate.loc[candidate_index]
    cutoff = row["measurement_date"]
    development = p95.loc[p95.measurement_date.le(cutoff)]
    test = p95.loc[p95.measurement_date.gt(cutoff)]
    train_households = set(development.household_id.astype(str))
    test_households = set(test.household_id.astype(str))
    unseen = test_households - train_households
    test_end = test.measurement_date.max()
    candidate_rows.append({
        "desired_development_fraction": desired_fraction, "cutoff_inclusive": cutoff,
        "score": float(row.score), "development_rows": len(development), "development_row_fraction": len(development) / total_rows,
        "test_rows": len(test), "test_row_fraction": len(test) / total_rows,
        "development_p95_events": int(development.target.sum()), "test_p95_events": int(test.target.sum()),
        "development_p90_events": int(p90.loc[p90.measurement_date.le(cutoff), "target"].sum()),
        "test_p90_events": int(p90.loc[p90.measurement_date.gt(cutoff), "target"].sum()),
        "development_p95_positive_fraction": float(row.dev_positive_fraction), "test_p95_positive_fraction": 1 - float(row.dev_positive_fraction),
        "development_p95_negatives": int((development.target == 0).sum()), "test_p95_negatives": int((test.target == 0).sum()),
        "development_unique_dates": int(development.measurement_date.nunique()), "test_unique_dates": int(test.measurement_date.nunique()),
        "test_calendar_span_days": int((test_end - cutoff).days) if len(test) else 0,
        "development_households": len(train_households), "test_households": len(test_households),
        "test_seen_households": len(test_households & train_households), "test_unseen_households": len(unseen),
        "test_unseen_rows": int(test.household_id.astype(str).isin(unseen).sum()),
    })
candidate_summary = pd.DataFrame(candidate_rows).sort_values("desired_development_fraction", ascending=False).reset_index(drop=True)
save_table(candidate_summary, "04_02_candidate_cutoff_summary.csv", "Candidate cutoff comparison")

cutoff_curve = date_counts[["measurement_date", "rows", "p95_positive", "p95_negative", "p90_positive", "p90_negative", "cum_rows", "cum_p95_positive", "cum_p95_negative", "cum_p90_positive"]].copy()
cutoff_curve["row_fraction"] = cutoff_curve.cum_rows / total_rows
cutoff_curve["p95_positive_fraction"] = cutoff_curve.cum_p95_positive / total_positive
cutoff_curve["p95_negative_fraction"] = cutoff_curve.cum_p95_negative / total_negative
cutoff_curve["p90_positive_fraction"] = cutoff_curve.cum_p90_positive / total_p90_positive
save_table(cutoff_curve, "04_03_cutoff_cumulative_curve.csv", "Cumulative rows and P95 classes by date")



## Stage 2 — freeze the outer cutoff, internal development blocks, and natural seen/unseen cohorts

The policy favours the largest development fraction that still leaves the configured minimum calendar test horizon and adequate P95 positive support. With these data, the mathematical 80% score-minimizer leaves only a short late-period test window; the notebook exposes that trade-off and selects the 70% candidate if it is the highest viable option. The same cutoff is applied to P95 and P90. Two additional whole-date boundaries assign outer-development rows near a 70%/15%/15% row split for model development, calibration, and threshold selection; both P95 and P90 event support and household counts are checked.


In [ ]:
viable = candidate_summary.loc[
    candidate_summary["test_calendar_span_days"].ge(MIN_TEST_CALENDAR_DAYS)
    & candidate_summary["test_p95_events"].ge(MIN_TEST_POSITIVE_EVENTS)
    & candidate_summary["development_p95_events"].ge(MIN_TEST_POSITIVE_EVENTS)
    & candidate_summary["test_p90_events"].ge(MIN_TEST_POSITIVE_EVENTS)
    & candidate_summary["development_p90_events"].ge(MIN_TEST_POSITIVE_EVENTS)
].sort_values("desired_development_fraction", ascending=False)
if viable.empty:
    raise RuntimeError("No candidate cutoff meets test-horizon and P95 positive-support requirements; revise the split protocol")
selected_candidate = viable.iloc[0]
SELECTED_CUTOFF = pd.Timestamp(selected_candidate["cutoff_inclusive"]).normalize()
SELECTED_DEVELOPMENT_FRACTION = float(selected_candidate["desired_development_fraction"])
print(f"Selected shared cutoff: {SELECTED_CUTOFF.date()} inclusive in development")
print(f"Selected desired development fraction: {SELECTED_DEVELOPMENT_FRACTION:.0%}; observed: {selected_candidate['development_row_fraction']:.2%}")
print(f"Test span: {int(selected_candidate['test_calendar_span_days'])} calendar days; P95 events: {int(selected_candidate['test_p95_events']):,}")

development_households = set(p95.loc[p95.measurement_date.le(SELECTED_CUTOFF), "household_id"].astype(str))
master = p95[KEY_COLUMNS].copy().rename(columns={"measurement_date": "target_date"})
master["outer_split"] = np.where(master["target_date"].le(SELECTED_CUTOFF), "development", "test")
master["seen_in_training"] = master["household_id"].astype(str).isin(development_households)
master["test_cohort"] = np.select(
    [master["outer_split"].eq("development"), master["seen_in_training"]],
    ["not_test", "test_seen"], default="test_unseen")

# Choose two whole-date boundaries within outer development by cumulative row counts.
# Candidate boundaries must leave enough dates and P95/P90 events in all three blocks.
dev_counts = date_counts.loc[date_counts.measurement_date.le(SELECTED_CUTOFF)].reset_index(drop=True).copy()
if len(dev_counts) < 3 * MIN_INTERNAL_BLOCK_DATES:
    raise RuntimeError("Outer development period has too few unique dates for internal blocks")
rows_cum = dev_counts.rows.to_numpy().cumsum()
p95_cum = dev_counts.p95_positive.to_numpy().cumsum()
p90_cum = dev_counts.p90_positive.to_numpy().cumsum()
rows_total_dev = int(rows_cum[-1])
p95_total_dev = int(p95_cum[-1])
p90_total_dev = int(p90_cum[-1])
internal_candidates = []
for first_idx in range(MIN_INTERNAL_BLOCK_DATES - 1, len(dev_counts) - 2 * MIN_INTERNAL_BLOCK_DATES):
    for second_idx in range(first_idx + MIN_INTERNAL_BLOCK_DATES, len(dev_counts) - MIN_INTERNAL_BLOCK_DATES):
        block_dates = [first_idx + 1, second_idx - first_idx, len(dev_counts) - second_idx - 1]
        if min(block_dates) < MIN_INTERNAL_BLOCK_DATES:
            continue
        row_counts = [int(rows_cum[first_idx]), int(rows_cum[second_idx] - rows_cum[first_idx]), int(rows_total_dev - rows_cum[second_idx])]
        p95_events = [int(p95_cum[first_idx]), int(p95_cum[second_idx] - p95_cum[first_idx]), int(p95_total_dev - p95_cum[second_idx])]
        p90_events = [int(p90_cum[first_idx]), int(p90_cum[second_idx] - p90_cum[first_idx]), int(p90_total_dev - p90_cum[second_idx])]
        if min(p95_events + p90_events) < MIN_INTERNAL_BLOCK_EVENTS:
            continue
        fractions = [count / rows_total_dev for count in row_counts]
        targets = list(INTERNAL_BLOCK_TARGET_FRACTIONS.values())
        score = sum(abs(actual - desired) for actual, desired in zip(fractions, targets))
        internal_candidates.append((score, first_idx, second_idx, row_counts, p95_events, p90_events, fractions))
if not internal_candidates:
    raise RuntimeError("No chronological internal split satisfies date and P95/P90 event support requirements")
internal_candidates.sort(key=lambda item: (item[0], item[1], item[2]))
selected_internal = None
for candidate in internal_candidates:
    _, first_idx, second_idx, *_ = candidate
    first_cut = pd.Timestamp(dev_counts.loc[first_idx, "measurement_date"]).normalize()
    second_cut = pd.Timestamp(dev_counts.loc[second_idx, "measurement_date"]).normalize()
    block_households = [
        p95.loc[p95.measurement_date.le(first_cut), "household_id"].nunique(),
        p95.loc[p95.measurement_date.gt(first_cut) & p95.measurement_date.le(second_cut), "household_id"].nunique(),
        p95.loc[p95.measurement_date.gt(second_cut) & p95.measurement_date.le(SELECTED_CUTOFF), "household_id"].nunique(),
    ]
    if min(block_households) >= MIN_INTERNAL_BLOCK_HOUSEHOLDS:
        selected_internal = candidate
        selected_internal_households = block_households
        break
if selected_internal is None:
    raise RuntimeError("No row-balanced internal split has adequate household support")
_, first_idx, second_idx, internal_rows, internal_p95_events, internal_p90_events, internal_fractions = selected_internal
MODEL_DEVELOPMENT_CUTOFF = pd.Timestamp(dev_counts.loc[first_idx, "measurement_date"]).normalize()
CALIBRATION_CUTOFF = pd.Timestamp(dev_counts.loc[second_idx, "measurement_date"]).normalize()
master["internal_split"] = np.select(
    [master.outer_split.eq("test"), master.target_date.le(MODEL_DEVELOPMENT_CUTOFF), master.target_date.le(CALIBRATION_CUTOFF)],
    ["not_applicable", "model_development", "calibration"], default="threshold_selection")
print("Internal whole-date cutoffs:", MODEL_DEVELOPMENT_CUTOFF.date(), "and", CALIBRATION_CUTOFF.date())
print("Internal row fractions:", {name: round(value, 3) for name, value in zip(INTERNAL_BLOCK_TARGET_FRACTIONS, internal_fractions)})

if master["outer_split"].eq("development").sum() == 0 or master["outer_split"].eq("test").sum() == 0:
    raise RuntimeError("Selected cutoff produced an empty partition")



## Stage 3 — build P95/P90 partition and cohort audits

`test_seen` is the primary known-household future-day cohort. `test_unseen` is a secondary natural cross-household cohort. A test household’s own earlier consumption may contribute to lag/rolling features; its labelled test rows are not used to fit model parameters.


In [ ]:
def target_split_summary(frame, target_name):
    assigned = frame[["row_id", "household_id", "measurement_date", "target"]].merge(
        master[["row_id", "outer_split", "seen_in_training", "test_cohort"]], on="row_id", how="left", validate="one_to_one")
    assigned["target_name"] = target_name
    return assigned

p95_assigned = target_split_summary(p95.rename(columns={"measurement_date": "measurement_date"}), "P95_primary")
p90_assigned = target_split_summary(p90.rename(columns={"measurement_date": "measurement_date"}), "P90_sensitivity")

split_summary = pd.concat([
    frame.groupby("outer_split", as_index=False).agg(rows=("row_id", "size"), households=("household_id", "nunique"),
        unique_dates=("measurement_date", "nunique"), positive_events=("target", "sum"), positive_rate=("target", "mean"))
        .assign(target=label)
    for label, frame in [("P95_primary", p95_assigned), ("P90_sensitivity", p90_assigned)]
], ignore_index=True)
save_table(split_summary, "04_04_split_summary_by_target.csv", "Shared split summary by target")

cohort_summary = p95_assigned.groupby(["outer_split", "test_cohort"], as_index=False).agg(
    rows=("row_id", "size"), households=("household_id", "nunique"), unique_dates=("measurement_date", "nunique"),
    p95_events=("target", "sum"), p95_rate=("target", "mean"))
save_table(cohort_summary, "04_05_seen_unseen_cohort_summary.csv", "P95 seen/unseen test cohorts")

calendar_summary = master.groupby("outer_split", as_index=False).agg(start_date=("target_date", "min"), end_date=("target_date", "max"),
    rows=("row_id", "size"), unique_dates=("target_date", "nunique"))
calendar_summary["calendar_span_days"] = (calendar_summary["end_date"] - calendar_summary["start_date"]).dt.days + 1
save_table(calendar_summary, "04_06_split_date_boundaries.csv", "Split date boundaries")



In [ ]:
internal_summary = master.loc[master.outer_split.eq("development")].merge(
    p95[["row_id", "target"]].rename(columns={"target": "p95_target"}), on="row_id", how="left", validate="one_to_one"
).merge(
    p90[["row_id", "target"]].rename(columns={"target": "p90_target"}), on="row_id", how="left", validate="one_to_one"
).groupby("internal_split", as_index=False).agg(
    start_date=("target_date", "min"), end_date=("target_date", "max"), unique_dates=("target_date", "nunique"),
    rows=("row_id", "size"), households=("household_id", "nunique"),
    p95_events=("p95_target", "sum"), p95_prevalence=("p95_target", "mean"),
    p90_events=("p90_target", "sum"), p90_prevalence=("p90_target", "mean"))
internal_summary["calendar_span_days"] = (internal_summary["end_date"] - internal_summary["start_date"]).dt.days + 1
internal_summary["row_fraction_of_outer_development"] = internal_summary["rows"] / len(p95.loc[p95.measurement_date.le(SELECTED_CUTOFF)])
internal_summary["target_row_fraction"] = internal_summary["internal_split"].map(INTERNAL_BLOCK_TARGET_FRACTIONS)
internal_summary = internal_summary.sort_values("start_date").reset_index(drop=True)
save_table(internal_summary, "04_08_internal_development_block_summary.csv", "Internal development block summary")

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
series = [
    ("row_fraction", "Cumulative row fraction", "Rows"),
    ("p95_positive_fraction", "Cumulative positive fraction", "P95 positives"),
    ("p90_positive_fraction", "Cumulative positive fraction", "P90 positives"),
]
for ax, (column, ylabel, label) in zip(axes, series):
    ax.plot(cutoff_curve.measurement_date, cutoff_curve[column], linewidth=1.8, color="#245c86", label=label)
    for cutoff, color, text in [
        (MODEL_DEVELOPMENT_CUTOFF, "#d97706", "Model development"),
        (CALIBRATION_CUTOFF, "#7c3aed", "Calibration"),
        (SELECTED_CUTOFF, "#b91c1c", "Outer test"),
    ]:
        ax.axvline(cutoff, color=color, linestyle="--", linewidth=1.2, label=f"{text}: {cutoff:%Y-%m-%d}")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper left", ncol=2, fontsize=8)
axes[-1].set_xlabel("Target date")
fig.suptitle("Cumulative rows and positive events with frozen chronological cutoffs", y=0.995)
fig.tight_layout()
figure_path = FIGURE_DIR / "04_01_cumulative_split_design.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
print("Cumulative split figure:", figure_path)


## Stage 4 — save one master split and verify it can be reused

The master file contains the stable join key, household and target date, outer partition, internal development role, and seen/unseen cohort metadata. Later M0/M1 datasets join this file on `row_id`; later notebooks must not create a second outer split.


In [ ]:
master = master[["row_id", "household_id", "target_date", "outer_split", "internal_split", "seen_in_training", "test_cohort"]].copy()
master = master.sort_values(["target_date", "household_id"], kind="mergesort").reset_index(drop=True)
write_csv_atomic(master, MASTER_SPLIT_PATH)
master_contract = write_dtype_contract(master, MASTER_SPLIT_PATH, MASTER_SPLIT_CONTRACT_PATH)

reloaded = read_contract_csv(MASTER_SPLIT_PATH, master_contract)
reloaded["household_id"] = reloaded["household_id"].astype("string")
reloaded["target_date"] = pd.to_datetime(reloaded["target_date"], errors="raise").dt.normalize()
reloaded = reloaded.sort_values(["target_date", "household_id"], kind="mergesort").reset_index(drop=True)
master_roundtrip_matches = (
    reloaded["row_id"].astype(str).equals(master["row_id"].astype(str))
    and reloaded["household_id"].astype(str).equals(master["household_id"].astype(str))
    and reloaded["target_date"].equals(master["target_date"])
    and reloaded["outer_split"].astype(str).equals(master["outer_split"].astype(str))
    and reloaded["internal_split"].astype(str).equals(master["internal_split"].astype(str))
    and reloaded["seen_in_training"].astype(bool).equals(master["seen_in_training"].astype(bool))
    and reloaded["test_cohort"].astype(str).equals(master["test_cohort"].astype(str))
)
date_partition_counts = master.groupby("target_date")["outer_split"].nunique()
p95_split = p95_assigned.groupby("outer_split")["target"].nunique()
p90_split = p90_assigned.groupby("outer_split")["target"].nunique()

final_checks = pd.DataFrame([
    ("all_input_checks_pass", bool(input_checks.passed.all()), str(len(input_checks))),
    ("same_common_cutoff_used", p95_assigned.merge(p90_assigned, on="row_id", suffixes=("_p95", "_p90"))["outer_split_p95"].equals(p95_assigned.merge(p90_assigned, on="row_id", suffixes=("_p95", "_p90"))["outer_split_p90"]), str(SELECTED_CUTOFF.date())),
    ("selected_test_horizon_meets_minimum", int(selected_candidate["test_calendar_span_days"]) >= MIN_TEST_CALENDAR_DAYS, str(int(selected_candidate["test_calendar_span_days"]))),
    ("test_and_development_p95_p90_support", all(int(candidate_summary.loc[candidate_summary.desired_development_fraction.eq(SELECTED_DEVELOPMENT_FRACTION), column].iloc[0]) >= MIN_TEST_POSITIVE_EVENTS for column in ["test_p95_events", "development_p95_events", "test_p90_events", "development_p90_events"]), str({column: int(selected_candidate[column]) for column in ["development_p95_events", "test_p95_events", "development_p90_events", "test_p90_events"]})),
    ("one_partition_per_target_date", date_partition_counts.max() == 1, str(int(date_partition_counts.max()))),
    ("development_precedes_test", master.loc[master.outer_split.eq("development"), "target_date"].max() < master.loc[master.outer_split.eq("test"), "target_date"].min(), f"cutoff={SELECTED_CUTOFF.date()}"),
    ("p95_both_classes_in_each_partition", p95_split.eq(2).all(), str(p95_split.to_dict())),
    ("p90_both_classes_in_each_partition", p90_split.eq(2).all(), str(p90_split.to_dict())),
    ("internal_blocks_chronological_and_complete", internal_summary.internal_split.tolist() == ["model_development", "calibration", "threshold_selection"] and internal_summary.rows.sum() == int(master.outer_split.eq("development").sum()), str(internal_summary[["internal_split", "start_date", "end_date", "rows"]].to_dict("records"))),
    ("internal_blocks_have_p95_p90_event_support", internal_summary[["p95_events", "p90_events"]].ge(MIN_INTERNAL_BLOCK_EVENTS).all().all(), str(internal_summary[["internal_split", "p95_events", "p90_events"]].to_dict("records"))),
    ("internal_blocks_have_household_support", internal_summary.households.ge(MIN_INTERNAL_BLOCK_HOUSEHOLDS).all(), str(internal_summary.set_index("internal_split").households.to_dict())),
    ("outer_test_has_no_internal_role", master.loc[master.outer_split.eq("test"), "internal_split"].eq("not_applicable").all(), "outer test rows marked not_applicable"),
    ("seen_unseen_cohorts_correct", (master.loc[master.outer_split.eq("test"), "test_cohort"].eq(np.where(master.loc[master.outer_split.eq("test"), "seen_in_training"], "test_seen", "test_unseen")).all()), "test household membership derived from development households"),
    ("master_row_id_unique", master.row_id.is_unique and len(master) == len(p95), f"{master.row_id.nunique():,}"),
    ("master_csv_roundtrip", master_roundtrip_matches, "values and order reconstructed across string/bool/date contract"),
    ("cumulative_split_figure_exists", figure_path.exists() and figure_path.stat().st_size > 0, str(figure_path)),
    ("master_dtype_contract_exists", MASTER_SPLIT_CONTRACT_PATH.exists(), str(MASTER_SPLIT_CONTRACT_PATH)),
    ("master_csv_only", MASTER_SPLIT_PATH.suffix.lower() == ".csv", MASTER_SPLIT_PATH.name),
    ("no_household_random_holdout", True, "household overlap is reported, not forced"),
], columns=["check", "passed", "evidence"])
save_table(final_checks, "04_07_final_validation.csv", "Split design final validation")
if not final_checks.passed.all(): raise RuntimeError(f"Split design validation failed: {final_checks.loc[~final_checks.passed, 'check'].tolist()}")



## Stage 5 — split manifest and handoff

The split policy is recorded with its candidate diagnostics. The cutoff is selected using P95 only and shared unchanged with P90. This natural time split evaluates future observations; most test rows may belong to households that also appear in development, so report the primary result as known-household future-day performance and the unseen cohort separately.


In [ ]:
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "stage": "04_split_design",
    "input_signatures": {key: {"path": str(path), "sha256": sha256(path), "size_bytes": path.stat().st_size} for key, path in paths.items()},
    "primary_target_for_cutoff": "P95", "sensitivity_target": "P90", "candidate_score": "|row_fraction-r| + |P95_positive_fraction-r| + |P95_negative_fraction-r|",
    "candidate_development_fractions": DESIRED_DEVELOPMENT_FRACTIONS,
    "minimum_test_calendar_days": MIN_TEST_CALENDAR_DAYS, "minimum_positive_events_per_target_partition": MIN_TEST_POSITIVE_EVENTS,
    "minimum_internal_block_events_per_target": MIN_INTERNAL_BLOCK_EVENTS,
    "internal_block_target_fractions": INTERNAL_BLOCK_TARGET_FRACTIONS,
    "internal_cutoffs_inclusive": {"model_development": str(MODEL_DEVELOPMENT_CUTOFF.date()), "calibration": str(CALIBRATION_CUTOFF.date()), "threshold_selection_ends": str(SELECTED_CUTOFF.date())},
    "internal_block_summary": internal_summary.to_dict("records"),
    "selected_desired_development_fraction": SELECTED_DEVELOPMENT_FRACTION,
    "selected_cutoff_inclusive_for_development": str(SELECTED_CUTOFF.date()),
    "observed_development_fraction": float(selected_candidate["development_row_fraction"]),
    "observed_test_fraction": float(selected_candidate["test_row_fraction"]),
    "test_calendar_span_days": int(selected_candidate["test_calendar_span_days"]),
    "development_p90_events": int(selected_candidate["development_p90_events"]), "test_p90_events": int(selected_candidate["test_p90_events"]),
    "split_figure": str(figure_path),
    "test_seen_households": int(selected_candidate["test_seen_households"]),
    "test_unseen_households": int(selected_candidate["test_unseen_households"]),
    "primary_evaluation_cohort": "test_seen", "secondary_generalization_cohort": "test_unseen",
    "master_split": {"path": str(MASTER_SPLIT_PATH), "rows": len(master), "columns": list(master.columns), "sha256": sha256(MASTER_SPLIT_PATH)},
    "dtype_contract": str(MASTER_SPLIT_CONTRACT_PATH), "validation": {row.check: bool(row.passed) for row in final_checks.itertuples(index=False)},
    "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
write_json(manifest, TABLE_DIR / "04_split_design_manifest.json")
print("Split design completed; all checks passed.")
print(f"Master split: {MASTER_SPLIT_PATH} ({len(master):,} rows)")
print(f"Development through {SELECTED_CUTOFF.date()}; test begins {master.loc[master.outer_split.eq('test'), 'target_date'].min().date()}.")
print(f"Test seen households={int(selected_candidate['test_seen_households'])}; unseen={int(selected_candidate['test_unseen_households'])}.")


### Important evaluation interpretation

The `test_seen` rows are future days for households represented in development; the model parameters are fitted only on development labels, while each household's prior consumption history remains available for its lagged features. `test_unseen` is the naturally occurring secondary cohort. A small unseen cohort should be reported as limited evidence, not as a definitive new-household generalization result. All later experiments join this master CSV by `row_id` and reuse the same outer and internal assignments. The internal blocks use whole dates and approximate 70%/15%/15% of outer-development rows; inspect the saved block table and cumulative plot before interpreting their calendar span.
